In [0]:
# ========== INSIGHTS E CASOS DE USO ==========

print("INSIGHTS E CASOS DE USO - SILVER_REVIEW_SENTIMENT")
print("="*80)

print("\nA nova tabela 'silver_review_sentiment' permite:")
print()
print("1. DETECÇÃO DE ANOMALIAS")
print("   - Identificar reviews com discrepância entre texto e nota")
print("   - Filtrar possíveis casos de sarcasmo ou reviews suspeitos")
print()
print("2. ANÁLISE DE QUALIDADE DOS REVIEWS")
print("   - Avaliar consistência dos usuários em suas avaliações")
print("   - Identificar padrões de avaliação por negócio")
print()
print("3. INSIGHTS DE NEGÓCIO")
print("   - Segmentar negócios por sentimento real vs percebido")
print("   - Priorizar ações em negócios com sentimento negativo")
print()
print("4. MACHINE LEARNING")
print("   - Feature para modelos de recomendação")
print("   - Treinamento de modelos de detecção de sentimento")
print()
print("="*80)

print(f"\nLocalização: {CATALOG}.{SCHEMA_SILVER}.silver_review_sentiment")
print("\nCampos principais:")
print("  - sentiment: sentimento detectado (positive/negative/neutral/mixed)")
print("  - sentiment_coherence: TRUE/FALSE indica se o sentimento é coerente com a nota")
print("  - expected_sentiment: sentimento esperado baseado na nota (stars)")
print("\n✓ Execute as células na ordem para criar e analisar a tabela!")

In [0]:
# ========== EXEMPLOS DE REVIEWS ==========

try:
    # Carrega a tabela se ainda não estiver carregada
    if 'df_sentiment' not in dir():
        df_sentiment = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_review_sentiment")

    print("EXEMPLOS DE REVIEWS - ANÁLISE QUALITATIVA")
    print("="*80)

    # Exemplos de reviews COERENTES
    print("\n1. EXEMPLOS DE REVIEWS COERENTES (sentimento = nota)")
    print("-"*80)
    df_coerentes = df_sentiment.filter(col('sentiment_coherence') == True).select(
        'review_id',
        'stars',
        'sentiment',
        'expected_sentiment',
        'text'
    ).limit(5)

    display(df_coerentes)

    # Exemplos de reviews INCOERENTES
    print("\n2. EXEMPLOS DE REVIEWS INCOERENTES (sentimento ≠ nota)")
    print("-"*80)
    print("Estes casos podem indicar:")
    print("  - Sarcasmo ou ironia no texto")
    print("  - Avaliações mistas (pontos positivos e negativos)")
    print("  - Notas que não refletem o conteúdo escrito\n")

    df_incoerentes = df_sentiment.filter(col('sentiment_coherence') == False).select(
        'review_id',
        'stars',
        'sentiment',
        'expected_sentiment',
        'text'
    ).limit(5)

    display(df_incoerentes)

    # Análise de incoerência por tipo
    print("\n3. ANÁLISE DE INCOERÊNCIA POR TIPO")
    print("-"*80)
    df_incoer_tipos = df_sentiment.filter(col('sentiment_coherence') == False) \
        .groupBy('stars', 'sentiment') \
        .count() \
        .orderBy(col('count').desc())

    print("Casos mais comuns de incoerência:")
    display(df_incoer_tipos)

    print("\n✓ Análise de exemplos concluída!")
    
except Exception as e:
    error_msg = str(e)
    if "TABLE_OR_VIEW_NOT_FOUND" in error_msg or "silver_review_sentiment" in error_msg:
        print("⚠ ERRO: A tabela 'silver_review_sentiment' ainda não foi criada.")
        print("\nExecute as células na seguinte ordem:")
        print("  1. Cell 4: Configuração (define CATALOG e SCHEMA_SILVER)")
        print("  2. Cell 5: Criação da Tabela (executa ai_analyze_sentiment)")
        print("  3. Esta célula: Exemplos de reviews")
    elif "NameError" in error_msg or "CATALOG" in error_msg or "SCHEMA_SILVER" in error_msg:
        print("⚠ ERRO: Variáveis de configuração não estão definidas.")
        print("\nExecute primeiro a Cell 4: Configuração")
    else:
        print(f"⚠ ERRO: {error_msg}")
        raise

In [0]:
# ========== DEMONSTRAÇÃO: ANÁLISE DE SENTIMENTO E COERÊNCIA ==========

print("ANÁLISE DE SENTIMENTO DOS REVIEWS - COERÊNCIA COM NOTAS")
print("="*80)

# Carrega a tabela
df_sentiment = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_review_sentiment")

# 1. Métricas Gerais
print("\n1. MÉTRICAS GERAIS")
print("-"*80)
total_reviews = df_sentiment.count()
print(f"Total de reviews analisados: {total_reviews:,}")

# 2. Distribuição de Sentimentos Detectados
print("\n2. DISTRIBUIÇÃO DE SENTIMENTOS DETECTADOS")
print("-"*80)
df_sent_dist = df_sentiment.groupBy('sentiment').count().orderBy(col('count').desc())
display(df_sent_dist)

# 3. Taxa de Coerência Geral
print("\n3. TAXA DE COERÊNCIA GERAL")
print("-"*80)
coerentes = df_sentiment.filter(col('sentiment_coherence') == True).count()
incoerentes = df_sentiment.filter(col('sentiment_coherence') == False).count()
taxa_coerencia = (coerentes / total_reviews * 100) if total_reviews > 0 else 0

print(f"Reviews COERENTES: {coerentes:,} ({taxa_coerencia:.2f}%)")
print(f"Reviews INCOERENTES: {incoerentes:,} ({(100 - taxa_coerencia):.2f}%)")

# 4. Coerência por Nota (Stars)
print("\n4. COERÊNCIA POR NOTA (STARS)")
print("-"*80)
df_coer_stars = df_sentiment.groupBy('stars', 'sentiment_coherence') \
    .count() \
    .orderBy('stars', col('sentiment_coherence').desc())
display(df_coer_stars)

# 5. Matriz de Confusão: Sentimento Esperado vs Detectado
print("\n5. MATRIZ: SENTIMENTO ESPERADO vs DETECTADO")
print("-"*80)
df_matriz = df_sentiment.groupBy('expected_sentiment', 'sentiment') \
    .count() \
    .orderBy('expected_sentiment', col('count').desc())
display(df_matriz)

print("\n✓ Análise concluída!")

In [0]:
# ========== CONFIGURAÇÃO ==========
from pyspark.sql.functions import *

# Definir catálogo e schema
CATALOG = "workspace"
SCHEMA_SILVER = "yelp_ing"

print(f"Catálogo: {CATALOG}")
print(f"Schema Silver: {SCHEMA_SILVER}")
print("\n✓ Configuração concluída!")

In [0]:
%sql
-- ========== CRIAÇÃO DA TABELA SILVER_REVIEW_SENTIMENT (OTIMIZADA) ==========
-- Análise de sentimento dos reviews e verificação de coerência com a nota
-- OTIMIZAÇÃO: Calcula ai_analyze_sentiment apenas UMA vez por linha

CREATE OR REPLACE TABLE workspace.yelp_ing.silver_review_sentiment AS
WITH sentiment_calculated AS (
  SELECT 
    r.*,
    -- Calcula o sentimento UMA ÚNICA VEZ
    ai_analyze_sentiment(r.text) AS sentiment
  FROM workspace.yelp_ing.silver_review r
  WHERE r.text IS NOT NULL AND LENGTH(TRIM(r.text)) > 0
)
SELECT 
  *,
  -- Determina se o sentimento é coerente com a nota (reutiliza o valor já calculado)
  CASE 
    -- Reviews com nota baixa (1-2 estrelas) devem ter sentimento negativo
    WHEN stars <= 2 AND sentiment = 'negative' THEN TRUE
    
    -- Reviews com nota média (3 estrelas) podem ser neutros ou mistos
    WHEN stars = 3 AND sentiment IN ('neutral', 'mixed') THEN TRUE
    
    -- Reviews com nota alta (4-5 estrelas) devem ter sentimento positivo
    WHEN stars >= 4 AND sentiment = 'positive' THEN TRUE
    
    -- Casos sem coerência (sentimento não corresponde à nota)
    ELSE FALSE
  END AS sentiment_coherence,
  
  -- Sentimento esperado baseado na nota
  CASE
    WHEN stars <= 2 THEN 'negative'
    WHEN stars = 3 THEN 'neutral'
    ELSE 'positive'
  END AS expected_sentiment,
  
  -- Metadados da análise
  current_timestamp() AS sentiment_analysis_timestamp
  
FROM sentiment_calculated;